# 디렉토리 구조, S3 구조
```
load_run_save_boilerplate.ipynb - SageMaker Training 보일러플레이트

Workflow:
    1. S3 Input → 로컬로 다운로드
    2. 모델링 수행 (목업)
    3. 결과물 → S3 Output 업로드

S3 Input 구조:
    s3://gs-retail-awesome-data-{region}/
        env={env}/user={user_id}/project={project}/version={version}/
            ├── config/
            ├── data/
            ├── metadata/
            └── profile/

S3 Output 구조:
    s3://gs-retail-awesome-model-{region}/
        env={env}/user={user_id}/project={project}/experiment={experiment}/
        model={model}/algo={algo}/run_date={run_date}/run_id={run_id}/
            ├── metadata/run_manifest.yml
            ├── config/config.yml
            ├── data_refs/input_data_ref.yml
            ├── artifacts/
            │   ├── model/.placeholder
            │   ├── metrics/model_metrics.yml
            │   ├── charts/.placeholder
            │   └── explainability/.placeholder
            └── reports/report_request.yml
```

In [1]:
# %% Imports
import os
import boto3
import yaml
import uuid
import getpass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

# %% Configuration
class InputConfig:
    """Input S3 설정 (데이터셋)"""
    
    REGION = "us-east-1"
    BUCKET = f"gs-retail-awesome-data-{REGION}"
    ENV = "dev"
    USER_ID = getpass.getuser()
    PROJECT = "titanic"
    VERSION = "v1.0"
    
    @classmethod
    def get_s3_prefix(cls, env=None, user_id=None, project=None, version=None):
        return (
            f"env={env or cls.ENV}/"
            f"user={user_id or cls.USER_ID}/"
            f"project={project or cls.PROJECT}/"
            f"version={version or cls.VERSION}"
        )
    
    @classmethod
    def get_s3_uri(cls, **kwargs):
        prefix = cls.get_s3_prefix(**kwargs)
        return f"s3://{cls.BUCKET}/{prefix}/"


class OutputConfig:
    """Output S3 설정 (모델 결과물)"""
    
    REGION = "us-east-1"
    BUCKET = f"gs-retail-awesome-model-{REGION}"
    ENV = "dev"
    USER_ID = getpass.getuser()
    PROJECT = "gs25-sales-forecast"
    EXPERIMENT = "baseline-v1"
    MODEL = "store-daily-sales-forecast"
    ALGO = "lightgbm"
    
    @classmethod
    def generate_run_id(cls):
        now = datetime.utcnow()
        short_uuid = uuid.uuid4().hex[:8]
        return now.strftime(f"%Y%m%dT%H%M%SZ_{short_uuid}")
    
    @classmethod
    def get_run_date(cls):
        return datetime.utcnow().strftime("%Y-%m-%d")
    
    @classmethod
    def get_s3_prefix(cls, env=None, user_id=None, project=None, experiment=None, 
                      model=None, algo=None, run_date=None, run_id=None):
        return (
            f"env={env or cls.ENV}/"
            f"user={user_id or cls.USER_ID}/"
            f"project={project or cls.PROJECT}/"
            f"experiment={experiment or cls.EXPERIMENT}/"
            f"model={model or cls.MODEL}/"
            f"algo={algo or cls.ALGO}/"
            f"run_date={run_date or cls.get_run_date()}/"
            f"run_id={run_id or cls.generate_run_id()}"
        )
    
    @classmethod
    def get_s3_uri(cls, **kwargs):
        prefix = cls.get_s3_prefix(**kwargs)
        return f"s3://{cls.BUCKET}/{prefix}/"


class LocalConfig:
    """로컬 디렉토리 설정"""
    
    ROOT = "."
    INPUT_DIR = "input"
    OUTPUT_DIR = "output"
    
    # Output 하위 폴더 구조
    OUTPUT_SUBDIRS = [
        "metadata",
        "config", 
        "data_refs",
        "artifacts/model",
        "artifacts/metrics",
        "artifacts/charts",
        "artifacts/explainability",
        "reports"
    ]

In [2]:
# %% Generic S3 Functions
def ensure_bucket_exists(bucket, region=None):
    """S3 버킷이 없으면 생성"""
    region = region or OutputConfig.REGION
    s3_client = boto3.client('s3', region_name=region)
    
    try:
        s3_client.head_bucket(Bucket=bucket)
        print(f"✓ Bucket exists: {bucket}")
        return True
    except s3_client.exceptions.ClientError as e:
        error_code = e.response.get('Error', {}).get('Code')
        
        if error_code in ['404', 'NoSuchBucket']:
            print(f"⚠ Bucket not found, creating: {bucket}")
            try:
                if region == 'us-east-1':
                    s3_client.create_bucket(Bucket=bucket)
                else:
                    s3_client.create_bucket(
                        Bucket=bucket,
                        CreateBucketConfiguration={'LocationConstraint': region}
                    )
                print(f"✓ Bucket created: {bucket} (region: {region})")
                return True
            except Exception as create_error:
                print(f"✗ Failed to create bucket: {create_error}")
                return False
        else:
            print(f"✗ Bucket access error: {e}")
            return False


def download_from_s3(bucket, s3_prefix, local_dir, dry_run=False):
    """S3에서 로컬로 다운로드 (generic)"""
    s3_client = boto3.client('s3')
    downloaded = []
    
    try:
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket, Prefix=s3_prefix)
        
        for page in pages:
            for obj in page.get('Contents', []):
                s3_key = obj['Key']
                relative_path = s3_key[len(s3_prefix):].lstrip('/')
                local_path = os.path.join(local_dir, relative_path)
                
                if dry_run:
                    print(f"  [DRY RUN] s3://{bucket}/{s3_key} -> {local_path}")
                else:
                    os.makedirs(os.path.dirname(local_path), exist_ok=True)
                    s3_client.download_file(bucket, s3_key, local_path)
                    print(f"  ✓ Downloaded: {local_path}")
                
                downloaded.append(local_path)
    except Exception as e:
        print(f"✗ Download error: {e}")
    
    return downloaded


def upload_to_s3(local_path, bucket, s3_key, dry_run=False):
    """단일 파일 S3 업로드 (generic)"""
    s3_uri = f"s3://{bucket}/{s3_key}"
    
    if dry_run:
        print(f"  [DRY RUN] {local_path} -> {s3_uri}")
    else:
        s3_client = boto3.client('s3')
        s3_client.upload_file(local_path, bucket, s3_key)
        print(f"  ✓ Uploaded: {s3_uri}")
    
    return s3_uri


def upload_directory_tree(local_base, bucket, s3_prefix, dry_run=False):
    """디렉토리 트리 전체를 S3에 업로드 (generic)"""
    uploaded = []
    
    for root, dirs, files in os.walk(local_base):
        for filename in files:
            local_path = os.path.join(root, filename)
            relative_path = os.path.relpath(local_path, local_base)
            s3_key = f"{s3_prefix}/{relative_path}"
            
            uri = upload_to_s3(local_path, bucket, s3_key, dry_run)
            uploaded.append(uri)
    
    return uploaded


# %% Generic File Functions
def create_directories(base_path, subdirs):
    """디렉토리 구조 생성 (generic)"""
    paths = {'base': base_path}
    os.makedirs(base_path, exist_ok=True)
    
    for subdir in subdirs:
        full_path = os.path.join(base_path, subdir)
        os.makedirs(full_path, exist_ok=True)
        paths[subdir] = full_path
    
    return paths


def create_file(filepath, content=None, file_type="yaml"):
    """파일 생성 (generic)"""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    
    if file_type == "touch" or content is None:
        Path(filepath).touch()
    elif file_type == "yaml":
        with open(filepath, 'w', encoding='utf-8') as f:
            yaml.dump(content, f, default_flow_style=False, allow_unicode=True)
    elif file_type == "json":
        import json
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(content, f, indent=2, ensure_ascii=False)
    else:
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(str(content))
    
    return filepath


# %% Output Mockup Definitions
def get_output_mockups(run_id, user_id=None):
    """Output 결과물 목업 데이터 정의"""
    return {
        'metadata': {
            'run_manifest.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'run_id': run_id,
                    'created_at': datetime.utcnow().isoformat(),
                    'created_by': user_id or OutputConfig.USER_ID,
                    'status': 'completed',
                    'project': OutputConfig.PROJECT,
                    'experiment': OutputConfig.EXPERIMENT,
                    'model': OutputConfig.MODEL,
                    'algo': OutputConfig.ALGO,
                    'execution': {
                        'instance_type': 'ml.m5.xlarge',
                        'instance_count': 1,
                        'duration_seconds': 1234,
                    }
                }
            }
        },
        'config': {
            'config.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'algorithm': {'name': 'lightgbm', 'task': 'regression'},
                    'hyperparameters': {
                        'objective': 'regression',
                        'metric': 'rmse',
                        'num_leaves': 31,
                        'learning_rate': 0.05,
                        'n_estimators': 1000,
                        'early_stopping_rounds': 50,
                    }
                }
            }
        },
        'data_refs': {
            'input_data_ref.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'input_s3_uri': InputConfig.get_s3_uri(user_id=user_id),
                    'data_sources': {
                        'train': {'format': 'parquet', 'rows': 100000},
                        'validation': {'format': 'parquet', 'rows': 10000},
                        'test': {'format': 'parquet', 'rows': 10000},
                    }
                }
            }
        },
        'artifacts/model': {
            '.placeholder': {'type': 'touch', 'content': None}
        },
        'artifacts/metrics': {
            'model_metrics.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'metrics': {
                        'train': {'rmse': 0.1234, 'mae': 0.0987, 'r2': 0.9123},
                        'validation': {'rmse': 0.1456, 'mae': 0.1123, 'r2': 0.8901},
                        'test': {'rmse': 0.1523, 'mae': 0.1198, 'r2': 0.8856},
                    },
                    'feature_importance': {
                        'day_of_week': 0.25,
                        'is_holiday': 0.20,
                        'temperature': 0.18,
                        'promotion_flag': 0.15,
                        'precipitation': 0.12,
                        'store_id': 0.10,
                    }
                }
            }
        },
        'artifacts/charts': {
            '.placeholder': {'type': 'touch', 'content': None}
        },
        'artifacts/explainability': {
            '.placeholder': {'type': 'touch', 'content': None}
        },
        'reports': {
            'report_request.yml': {
                'type': 'yaml',
                'content': {
                    'version': '1.0',
                    'report_type': 'model_training_summary',
                    'requested_at': datetime.utcnow().isoformat(),
                    'requested_by': user_id or OutputConfig.USER_ID,
                    'include_sections': [
                        'executive_summary',
                        'data_overview',
                        'model_performance',
                        'feature_importance',
                        'recommendations'
                    ]
                }
            }
        }
    }


def create_output_mockups(output_dir, mockups):
    """Output 목업 파일들 생성"""
    created_files = []
    
    for subdir, files in mockups.items():
        subdir_path = os.path.join(output_dir, subdir)
        os.makedirs(subdir_path, exist_ok=True)
        
        for filename, file_info in files.items():
            filepath = os.path.join(subdir_path, filename)
            content = file_info.get('content')
            file_type = file_info.get('type', 'yaml')
            create_file(filepath, content, file_type)
            created_files.append(filepath)
            print(f"  ✓ Created: {filepath}")
    
    return created_files

In [3]:
# %% Main Workflow Functions
def load_input(user_id=None, version=None, local_root=None, dry_run=False):
    """
    Step 1: S3 Input → 로컬로 다운로드
    """
    local_root = local_root or LocalConfig.ROOT
    local_input_dir = os.path.join(local_root, LocalConfig.INPUT_DIR)
    
    s3_prefix = InputConfig.get_s3_prefix(user_id=user_id, version=version)
    s3_uri = InputConfig.get_s3_uri(user_id=user_id, version=version)
    
    print("=" * 60)
    print("Step 1: S3 Input 다운로드")
    print("=" * 60)
    print(f"Source: {s3_uri}")
    print(f"Target: {local_input_dir}")
    print("-" * 60)
    
    os.makedirs(local_input_dir, exist_ok=True)
    downloaded = download_from_s3(InputConfig.BUCKET, s3_prefix, local_input_dir, dry_run)
    
    return {
        's3_uri': s3_uri,
        'local_dir': local_input_dir,
        'downloaded_files': downloaded
    }


def run_model(local_root=None):
    """
    Step 2: 모델링 수행 (목업 - 실제로는 스킵)
    """
    print("\n" + "=" * 60)
    print("Step 2: 모델링 수행 (목업)")
    print("=" * 60)
    print("  ... 모델 학습 중 (스킵) ...")
    print("  ... 검증 중 (스킵) ...")
    print("  ... 테스트 중 (스킵) ...")
    print("✓ 모델링 완료 (목업)")
    
    return {'status': 'completed', 'message': 'Mockup training completed'}


def save_output(user_id=None, run_id=None, run_date=None, local_root=None, dry_run=False):
    """
    Step 3: 결과물 → S3 Output 업로드
    """
    local_root = local_root or LocalConfig.ROOT
    local_output_dir = os.path.join(local_root, LocalConfig.OUTPUT_DIR)
    
    # run_id 생성 (없으면)
    run_id = run_id or OutputConfig.generate_run_id()
    run_date = run_date or OutputConfig.get_run_date()
    
    s3_prefix = OutputConfig.get_s3_prefix(user_id=user_id, run_date=run_date, run_id=run_id)
    s3_uri = OutputConfig.get_s3_uri(user_id=user_id, run_date=run_date, run_id=run_id)
    
    print("\n" + "=" * 60)
    print("Step 3: 결과물 생성 및 S3 업로드")
    print("=" * 60)
    print(f"Run ID: {run_id}")
    print(f"Run Date: {run_date}")
    print(f"Target: {s3_uri}")
    print("-" * 60)
    
    # 로컬 output 디렉토리 생성
    print("\n[로컬 결과물 생성]")
    os.makedirs(local_output_dir, exist_ok=True)
    mockups = get_output_mockups(run_id, user_id)
    created_files = create_output_mockups(local_output_dir, mockups)
    
    # S3 업로드
    print("\n[S3 업로드]")
    if not dry_run:
        if not ensure_bucket_exists(OutputConfig.BUCKET, OutputConfig.REGION):
            raise Exception(f"Failed to ensure bucket exists: {OutputConfig.BUCKET}")
    
    uploaded = upload_directory_tree(local_output_dir, OutputConfig.BUCKET, s3_prefix, dry_run)
    
    return {
        's3_uri': s3_uri,
        'run_id': run_id,
        'run_date': run_date,
        'local_dir': local_output_dir,
        'created_files': created_files,
        'uploaded_files': uploaded
    }


def load_run_save(user_id=None, version=None, local_root=None, dry_run=False):
    """
    전체 워크플로우 실행: Load → Run → Save
    """
    print("\n" + "=" * 70)
    print("  SageMaker Training Boilerplate - Load / Run / Save")
    print("=" * 70)
    print(f"User: {user_id or InputConfig.USER_ID}")
    print(f"Input Version: {version or InputConfig.VERSION}")
    print(f"Dry Run: {dry_run}")
    print("=" * 70)
    
    # Step 1: Load
    input_result = load_input(user_id=user_id, version=version, local_root=local_root, dry_run=dry_run)
    
    # Step 2: Run (Mockup)
    run_result = run_model(local_root=local_root)
    
    # Step 3: Save
    output_result = save_output(user_id=user_id, local_root=local_root, dry_run=dry_run)
    
    print("\n" + "=" * 70)
    print("완료!")
    print("=" * 70)
    print(f"Input:  {input_result['s3_uri']}")
    print(f"Output: {output_result['s3_uri']}")
    print("=" * 70)
    
    return {
        'input': input_result,
        'run': run_result,
        'output': output_result
    }

In [4]:
# %% Example Usage
# 전체 워크플로우 (dry_run)
# result = load_run_save(
#     user_id="sean",
#     version="v1.0",
#     dry_run=True
# )

In [5]:
# 실제 실행
result = load_run_save(
    user_id="sean",
    version="v1.0",
    dry_run=False
)


  SageMaker Training Boilerplate - Load / Run / Save
User: sean
Input Version: v1.0
Dry Run: False
Step 1: S3 Input 다운로드
Source: s3://gs-retail-awesome-data-us-east-1/env=dev/user=sean/project=titanic/version=v1.0/
Target: ./input
------------------------------------------------------------
  ✓ Downloaded: ./input/config/config.yml
  ✓ Downloaded: ./input/data/input_data_ref.yml
  ✓ Downloaded: ./input/data/test.parquet
  ✓ Downloaded: ./input/data/train.parquet
  ✓ Downloaded: ./input/data/validation.parquet
  ✓ Downloaded: ./input/metadata/run_manifest.yml
  ✓ Downloaded: ./input/profile/profile.yml

Step 2: 모델링 수행 (목업)
  ... 모델 학습 중 (스킵) ...
  ... 검증 중 (스킵) ...
  ... 테스트 중 (스킵) ...
✓ 모델링 완료 (목업)

Step 3: 결과물 생성 및 S3 업로드
Run ID: 20260303T142446Z_6df3ac7d
Run Date: 2026-03-03
Target: s3://gs-retail-awesome-model-us-east-1/env=dev/user=sean/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2026-03-03/run_id=20260303T142446Z_6df3

/tmp/ipykernel_26862/2783816859.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/tmp/ipykernel_26862/2783816859.py:57: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().strftime("%Y-%m-%d")
/tmp/ipykernel_26862/3941452453.py:137: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat(),
/tmp/ipykernel_26862/3941452453.py:219: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use 

✓ Bucket created: gs-retail-awesome-model-us-east-1 (region: us-east-1)
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/env=dev/user=sean/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2026-03-03/run_id=20260303T142446Z_6df3ac7d/config/config.yml
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/env=dev/user=sean/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2026-03-03/run_id=20260303T142446Z_6df3ac7d/reports/report_request.yml
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/env=dev/user=sean/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2026-03-03/run_id=20260303T142446Z_6df3ac7d/artifacts/explainability/.placeholder
  ✓ Uploaded: s3://gs-retail-awesome-model-us-east-1/env=dev/user=sean/project=gs25-sales-forecast/experiment=baseline-v1/model=store-daily-sales-forecast/algo=lightgbm/run_date=2

In [6]:
!tree

.
├── __init__.py
├── __pycache__
│   ├── conf.cpython-312.pyc
│   └── run_pm_utils.cpython-312.pyc
├── conf.py
├── input
│   ├── config
│   │   └── config.yml
│   ├── data
│   │   ├── input_data_ref.yml
│   │   ├── test.parquet
│   │   ├── train.parquet
│   │   └── validation.parquet
│   ├── metadata
│   │   └── run_manifest.yml
│   └── profile
│       └── profile.yml
├── output
│   ├── artifacts
│   │   ├── charts
│   │   ├── explainability
│   │   ├── metrics
│   │   │   └── model_metrics.yml
│   │   └── model
│   ├── config
│   │   └── config.yml
│   ├── data_refs
│   │   └── input_data_ref.yml
│   ├── metadata
│   │   └── run_manifest.yml
│   └── reports
│       └── report_request.yml
├── prepare_datasets.ipynb
├── run_pm.ipynb
└── run_pm_utils.py

16 directories, 19 files
